# Prediction code for modeling turn-taking vs. continuing speech

In [ ]:
""" LOAD DEPENDENCIES """

# general utilities
import math
import random
import itertools
import csv

# data handling
import pandas as pd
import numpy as np
from scipy import stats

# display utilities
from IPython.display import clear_output

# pre-processing
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing

# models & evaluations
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# general Scikit-Learn
import sklearn

In [ ]:
""" HELPER FUNCTION FOR CREATING SPEECH BEHAVIOR REGEX """ 
def GetSpeechRegex(encodesAsSeq, length, regex_base, minIdx=2):
    labels = ["speech","user_has_not_spoken"]
    regex = regex_base

    if encodesAsSeq:
        if length >= minIdx:
            labels.append("user_prior_speaker_seq_"+str(length)+"_")
    else:
        
        # processing speaker indices to keep
        for i in range(minIdx,length+1):
            labels.append("user_prior_speaker_idx_"+str(i)+"_")
            
    # generate regex 
    for label in labels:
        regex += "|"+label
    if (regex_base == ""):
        regex = regex[1:]
    return "(" + regex + ")"

## Segment dataset for modeling turn-taking vs. continuing speech

In [ ]:
""" LOAD IN DATASET """
df_raw = pd.read_csv("./aggregated-dataset.csv", header=0)
df_raw['label'] = df_raw['label'].map({True: 1, False: 0})

# filter by cutoff threshold
speech_duration_threshold = 0.323 # second as cutoff for invalid turn-transition label
df_raw = df_raw.loc[df_raw['duration'] > speech_duration_threshold]

""" PROCESS DATASET """
df_raw_positive = df_raw.loc[(df_raw['trueLabelCategory'] == "clean_turn_transition")]
df_raw_positive = df_raw_positive.loc[df_raw_positive['label'] == 1]
df_raw_positive['label'] = 1

df_raw_negative = df_raw.loc[(df_raw['trueLabelCategory'] == "continuing_speech")]
df_raw_negative = df_raw_negative.loc[df_raw_negative['label'] == 0]
df_raw_negative = df_raw_negative.loc[df_raw_negative['sublabel'] == "other_user_at_window"] # samples for non-speakers at window
df_raw_negative['label'] = 0

df_raw_balanced = pd.concat([df_raw_negative,df_raw_positive])

# remove features to prevent colinearality
df_raw_balanced = df_raw_balanced[df_raw_balanced.columns.drop(list(df_raw_balanced.filter(regex='mutual_gaze|user_head_pos_x_|user_head_pos_z_|user_head_rot_yaw_|ref_user_head_pos_x_|ref_user_head_pos_z_|ref_user_head_rot_yaw_')))]

# create week_group feature
df_raw_balanced["week_group"] =  df_raw_balanced['week'].astype(str) + '_' + df_raw_balanced['group'].astype(str)

# filter by group size
df_cleaned_balanced = df_raw_balanced.loc[(df_raw_balanced['group_size'] > 2)]

# create balanced dataset
df_filtered_positive = df_cleaned_balanced.loc[df_cleaned_balanced['label'] == 1]
df_filtered_negative = df_cleaned_balanced.loc[df_cleaned_balanced['label'] == 0]

if (len(df_filtered_positive) <= len(df_filtered_negative)):
    df_filtered_negative = df_filtered_negative.sample(n=len(df_filtered_positive), random_state=1230)
    df_filtered_balanced = pd.concat([df_filtered_positive,df_filtered_negative])
else:
    df_filtered_positive = df_filtered_positive.sample(n=len(df_filtered_negative), random_state=1230)
    df_filtered_balanced = pd.concat([df_filtered_positive,df_filtered_negative])

# create dummy variables for the speech sequence variables
prior_speaker_colnames = list(df_filtered_balanced.columns)
prior_speaker_colnames = [name for name in prior_speaker_colnames if "user_prior_speaker_idx_" in name]

for prior_speaker_colname in prior_speaker_colnames:
    df_dummies = pd.get_dummies(df_filtered_balanced[prior_speaker_colname],
                                          prefix = prior_speaker_colname,
                                          dtype=float,
                                          drop_first = True)
    df_filtered_balanced = pd.concat([df_filtered_balanced, df_dummies], axis=1)      
    df_filtered_balanced.drop([prior_speaker_colname], inplace=True, axis=1)

## Evaluation of Models

In [ ]:
""" HELPER FUNCTION FOR TRAINING AND EVALUATING MODELS """ 
def TrainAndEvaluateModel (model_choice, cv_type):
    
    """ initialize model """
    if model_choice == "lr":
        m = LogisticRegression(max_iter = 3000, 
                               solver = 'newton-cholesky', 
                               random_state = 1230,
                               penalty = None)
    elif model_choice == "rf":
        m = RandomForestClassifier(n_estimators = 300, 
                                   max_leaf_nodes = 500,
                                   max_depth = 30,
                                   min_samples_leaf = 5,
                                   min_samples_split = 5,
                                   random_state = 1230)
    elif model_choice == "mlp":
        m = MLPClassifier(random_state = 1230, 
                          hidden_layer_sizes=(3,), 
                          max_iter=300, 
                          learning_rate_init = 0.01,
                          verbose=True, 
                          learning_rate = "adaptive")
        
    elif model_choice == "gbc":
        m = GradientBoostingClassifier(n_estimators = 500,
                               learning_rate = 0.05,
                               max_depth = 5,
                               max_features = 0.25,
                               min_samples_split = 0.01,
                               subsample = 0.8,
                               random_state = 1230)
    
    """ training and evaluation """
    random.seed(1230)
    scaler = preprocessing.StandardScaler()
    
    """ set up regex for feature filtering """
    base_regex = "_ave|_min|_max|group_size|gaze|^user_big5|^ref_user_big5|^group_big5"
    base_regex = GetSpeechRegex(encodesAsSeq = False,length = 10,regex_base = base_regex)
    
    if cv_type == "all":
        """ training on all data """
        train_data = df_filtered_balanced # all data
        train_x = train_data.filter(regex=base_regex).copy()
        train_y = train_data['label']
        
        # rescale training data that are not categorical or dummy variables
        transformed_cols = train_x.columns[train_x.columns != 'user_has_not_spoken']
        transformed_cols = [col for col in transformed_cols if "user_is_ref_user" != col]
        transformed_cols = [col for col in transformed_cols if "ethnicity_" not in col]
        transformed_cols = [col for col in transformed_cols if 'user_prior_speaker_' not in col]
        
        scaler.fit(train_x[transformed_cols])
        train_x[transformed_cols] = scaler.transform(train_x[transformed_cols])
        train_x.fillna(0, inplace=True)
        
        fitted_m = log_reg_m.fit(train_x,train_y)
        print("model choice:",model_choice,"\tcv_type:",cv_type)
        print("\tROC AUC score:",roc_auc_score(train_y, fitted_m.predict_proba(train_x)[:, 1]))
    else:
        
        """ cross-validation evaluation """
        fold = 10
        idx = 0
    
        # setting up cv
        if cv_type == "week_group":
            unique_week_groups = df_filtered_balanced["week_group"].unique()
            random.shuffle(unique_week_groups)
            batch_size = math.floor(len(unique_week_groups)/fold)
        elif cv_type == "group":
            unique_groups = df_raw_balanced["group"].unique()
            random.shuffle(unique_groups)
            batch_size = math.floor(len(unique_groups)/fold)
        elif cv_type == "week":
            fold = 4
            batch_size = 1
    
        """ calculate auc_roc using given cv method """
        train_auc_roc = []
        test_auc_roc = []
        random.seed(1230)
        scaler = preprocessing.StandardScaler()
        
        fold_count = 0
        while fold_count < fold:
            fold_count += 1
            
            clear_output(wait=True)
            print("Processing fold count " + str(fold_count)+"/"+str(fold)+"...")
        
            # create training and testing data
            if cv_type == "week_group":
                if (fold_count == fold):
                    test_data = df_filtered_balanced.loc[df_filtered_balanced['week_group'].isin(unique_week_groups[idx:len(unique_week_groups)-1])]
                    train_data = df_filtered_balanced.loc[~df_filtered_balanced['week_group'].isin(unique_week_groups[idx:len(unique_week_groups)-1])]
                else:
                    test_data = df_filtered_balanced.loc[df_filtered_balanced['week_group'].isin(unique_week_groups[idx:min(idx+batch_size, len(unique_week_groups)-1)])]
                    train_data = df_filtered_balanced.loc[~df_filtered_balanced['week_group'].isin(unique_week_groups[idx:min(idx+batch_size, len(unique_week_groups)-1)])]
            elif cv_type == "group":
                if (fold_count == fold):
                    test_data = df_filtered_balanced.loc[df_filtered_balanced['group'].isin(unique_groups[idx:len(unique_groups)-1])]
                    train_data = df_filtered_balanced.loc[~df_filtered_balanced['group'].isin(unique_groups[idx:len(unique_groups)-1])]
                else:
                    test_data = df_filtered_balanced.loc[df_filtered_balanced['group'].isin(unique_groups[idx:min(idx+batch_size, len(unique_groups)-1)])]
                    train_data = df_filtered_balanced.loc[~df_filtered_balanced['group'].isin(unique_groups[idx:min(idx+batch_size, len(unique_groups)-1)])]
            elif cv_type == "week":
                test_data = df_filtered_balanced.loc[df_filtered_balanced['week'] == fold_count+2] # week index starts at 3
                train_data = df_filtered_balanced.loc[~(df_filtered_balanced['week'] == fold_count+2)]
                
            # select features for training data
            train_x = train_data.filter(regex=base_regex).copy()
            train_y = train_data['label']
        
            # rescale training data that are not categorical or dummy variables
            transformed_cols = train_x.columns[train_x.columns != 'user_has_not_spoken']
            transformed_cols = [col for col in transformed_cols if "user_is_ref_user" != col]
            transformed_cols = [col for col in transformed_cols if "ethnicity_" not in col]
            transformed_cols = [col for col in transformed_cols if 'user_prior_speaker_' not in col]

            scaler.fit(train_x[transformed_cols])
            train_x[transformed_cols] = scaler.transform(train_x[transformed_cols])
        
            # select features for testing data
            test_x = test_data.filter(regex=base_regex).copy()
            test_y = test_data['label']
        
            # rescale testing data that are not categorical or dummy variables
            test_x[transformed_cols] = scaler.transform(test_x[transformed_cols])
        
            # swap out NAs for 0s
            train_x.fillna(0, inplace=True)
            test_x.fillna(0, inplace=True)
            
            print("training size", len(train_x))
            print("testing size", len(test_x))
        
            # fit model on training data
            fitted_m = m.fit(train_x,train_y)
        
            # calculate roc auc for model
            train_auc_roc.append(roc_auc_score(train_y, fitted_m.predict_proba(train_x)[:, 1]))
            test_auc_roc.append(roc_auc_score(test_y, fitted_m.predict_proba(test_x)[:, 1]))
            
            idx += batch_size
            
        clear_output(wait=True)
        print("model choice:",model_choice,"\tcv_type:",cv_type)
        print("\tTraining AUC ROC:", np.mean(train_auc_roc), "("+str(np.std(train_auc_roc))+")")
        print("\tTesting AUC ROC:", np.mean(test_auc_roc), "("+str(np.std(test_auc_roc))+")")
    
        if (cv_type == "week"):
            print("\tTrained on week 1-3, tested on week 4:", test_auc_roc[-1])

In [ ]:
""" EVALUATION ACROSS MODELS AND CROSS-VALIDATION METHODS """
model_choice = "gbc" # lr, rf, gbc, and mlp
cv_type = 'week' # week_group, week, group

TrainAndEvaluateModel(model_choice, cv_type)